# GT-MedRAG — Notebook 03: DPO Extractor Agent (GPT-2 scale)

DPO-trains the **extractor agent** (paper Sec. 3.2, Eq. 7) to produce concise, informative reasoning triplets `(condition; relation; conclusion)` from each evidence snippet.

The paper builds preference pairs with a 70B-class reward model (Seva) + downstream respondent accuracy (Sacc) + length (Slen). At GPT-2 scale we substitute deterministic proxies that the paper's goal implicit - scoring triplets by **lexical support of the source snippet (faithfulness), relation-cue coverage (informativeness) and length (concision, the Slen analogue)**:

score(T) = 0.5 * S_facsimile + 0.3 * S_info + 0.2 * S_len

Training pairs come from `data/processed/triplet_pairs.jsonl` (built locally by `scripts/make_sft_data.py`), which already contains `chosen` / `rejected` pairs derived from gold seed-reference triplets. This notebook additionally computes the proxy scorer on sampled triplets for a larger pair set.

In [ ]:
!nvidia-smi -L
!pip -q install transformers datasets accelerate trl peft

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
REPO = '/content/medical-cdss'
DRIVE = '/content/drive/MyDrive/medical-cdss'
BASE = f'{DRIVE}/gpt2-clinical-base'   # phase-1 checkpoint
OUT = f'{DRIVE}/gpt2-clinical-extractor'
if not os.path.isdir(REPO):
    !git clone https://<REPLACE-WITH-YOUR-ACCOUNT>/medical-cdss.git $REPO
%cd $REPO

In [ ]:
import json, re
from pathlib import Path
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

PAIRS_FILE = Path(f'{REPO}/data/processed/triplet_pairs.jsonl')
assert PAIRS_FILE.exists(), 'run scripts/make_sft_data.py locally and upload triplet_pairs.jsonl'
pairs = [json.loads(l) for l in open(PAIRS_FILE)]
print(len(pairs), 'preference pairs')

# proxy scorer (paper's Seva+Slen at resource-frugal scale)
REL_CUES = ('suggested by', 'associated with', 'is diagnostic of', 'confirms',
            'indicates', 'predicts', 'confirmed by')
def proxy_score(triplet_text: str, snippet: str) -> float:
    m = re.search(r'\(([^;()]+);([^;()]+);([^;()]+)\)', triplet_text)
    if not m:
        return 0.0
    cond, rel, concl = [g.strip().lower() for g in m.groups()]
    snip = re.sub(r'[^a-z0-9 ]', ' ', snippet.lower())
    sup = sum(w in snip for w in cond.split()[:8]) / max(1, len(cond.split()[:8]))
    sup2 = sum(w in snip for w in concl.split()[:8]) / max(1, len(concl.split()[:8]))
    s_faithfulness = 0.5 * sup + 0.5 * sup2
    s_info = float(any(cue in rel for cue in REL_CUES))
    s_len = 1.0 - min(1.0, abs(len(triplet_text) - 120) / 200)
    return 0.5*s_faithfulness + 0.3*s_info + 0.2*s_len

# extend pair set: rescore chosen vs rejected using the proxy (deterministic)
rows = []
for rec in pairs:
    sc_chosen = proxy_score(rec['chosen'], rec['snippet'])
    sc_rejected = proxy_score(rec['rejected'], rec['snippet'])
    chosen, rejected = (rec['chosen'], rec['rejected']) if sc_chosen >= sc_rejected else (rec['rejected'], rec['chosen'])
    rows.append({
        'prompt': f"You are a medical reasoning assistant. Extract a reasoning triplet (condition; relation; conclusion) from the snippet.\nSnippet: {rec['snippet']}\nTriplet:",
        'chosen': chosen,
        'rejected': rejected,
        'score_chosen': max(sc_chosen, sc_rejected),
    })

# data augmentation: sample G=2 perturbed 'rejected' variants per snippet
aug = []
for rec in rows[:]:
    cond = re.search(r'\(([^;()]+);', rec['chosen'])
    if not cond:
        continue
    bad = f'({cond.group(1)[:8]}; uncertain; plausible diagnosis)'
    if proxy_score(bad, rec['prompt']) < rec['score_chosen']:
        aug.append({**rec, 'rejected': bad})
rows.extend(aug)
print(len(rows), 'total preference pairs after augmentation')
ds = Dataset.from_list([{'prompt': r['prompt'], 'chosen': r['chosen'], 'rejected': r['rejected']} for r in rows])
split = ds.train_test_split(test_size=0.15, seed=19)
split

In [ ]:
from trl import DPOConfig, DPOTrainer

tok = AutoTokenizer.from_pretrained(BASE)
tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(BASE)
ref_model = AutoModelForCausalLM.from_pretrained(BASE)

cfg = DPOConfig(
    output_dir=f'{DRIVE}/ckpt-dpo-extractor',
    beta=0.1,                                   # paper Sec. 4.1.2
    learning_rate=1e-5,                         # paper's extractor LR
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    fp16=True,
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    max_length=512,
    max_prompt_length=384,
    report_to='none',
)
trainer = DPOTrainer(model=model, ref_model=ref_model, args=cfg,
                     train_dataset=split['train'],
                     eval_dataset=split['test'],
                     processing_class=tok)
trainer.train()
trainer.save_model(OUT)
tok.save_pretrained(OUT)
print('DPO extractor saved ->', OUT)

In [ ]:
# smoke test: extractor on one evidence snippet
import json
from transformers import pipeline
gen = pipeline('text-generation', model=OUT, device=0)
snippet = json.loads(open(f"{REPO}/knowledge_base/chunks.jsonl").readline())['text']
prompt = ("You are a medical reasoning assistant. Extract a reasoning triplet "
          "(condition; relation; conclusion) from the snippet.\n"
          f"Snippet: {snippet[:800]}\nTriplet:")
print(gen(prompt, max_new_tokens=90, do_sample=True, temperature=0.6)[0]['generated_text'][-250:])

Next: notebook 04 improves the respondent with Grounded-GRPO.